# HydroFragments metrics walkthrough

This notebook tours the four metric families HydroFragments computes, one section each, on a slightly richer synthetic reach than `01_quickstart.ipynb`: **two pools** that breathe with the seasons and sometimes merge into one connected water body through a central channel. If you have not run `01_quickstart.ipynb` yet, start there first for the basic pipeline shape.

**The four families** (plain-language names; each maps to one or more of the canonical `metric_family` values in `hydrofragments.schema.MetricFamily`):

1. **Extent & Persistence** -- how much water, how reliably (`metric_family` in `{"extent", "persistence"}`).
2. **Morphology & Fragmentation** -- how the water is shaped and broken up (`metric_family` in `{"morphology", "fragmentation"}`).
3. **Clustering & Connectivity** -- how pools relate in space and network (`metric_family == "clustering"` today -- see the note in that section for why `"connectivity"` itself does not yet appear in real `analyze()` output).
4. **Dynamics** -- how it changes through the hydrological year (`metric_family == "dynamics"`).

We also run the **4-zone spatial stratification** (`build_zones`) before the family sections, since zones are the spatial lens the metrics above are commonly read through.

## Building the walkthrough reach

The fixture (`examples/_fixtures.py::walkthrough_water_timeseries`) is a 3-year, 7x25 pixel synthetic reach: two square pools, one at each end, that grow and shrink together following a repeating 12-month wet/dry pattern, connected only by a single-pixel-wide channel row that wets in the wettest ~40% of months. That is enough structure to give every metric family below something real to show, while staying small enough to compute in seconds.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np

import _fixtures as fx
from hydrofragments import HydroConfig, analyze, open_water_cube
from hydrofragments.models import AnalysisInputs

water = fx.walkthrough_water_timeseries()
cube = open_water_cube(water, input_kind="generic_binary")
water.sum(dim=("y", "x")).to_series().head(12)

### Running `analyze` across all four families at once

One `analyze()` call selects every profile we need (`fx.walkthrough_config` sets `metric_profiles: [contracts_core, pixel_temporal, channel, dynamics]`). The `channel` and `dynamics` profiles need extra inputs beyond the water cube itself -- a real drainage line (`AnalysisInputs.drainage`, `channel_wet_profiles`, `channel_segment_lengths_m`) and a hydrological-year extent series plus dual APSEC composites (`hydroyear_extent`, `max_water_apsec`, `median_apsec`). The helpers below build all of that from the same fixture -- see their docstrings in `examples/_fixtures.py` for exactly how (and for the dynamics inputs specifically, an explicit caveat that they are synthetic stand-ins, not a real dual-composite pipeline).

In [ ]:
context = fx.walkthrough_channel_context()
extent = fx.walkthrough_hydroyear_extent(water)
max_apsec, median_apsec = fx.walkthrough_apsec_composites(water, extent)

config = HydroConfig.from_mapping(fx.walkthrough_config(output_dir="metrics_walkthrough_out"))

channel_wet_profiles = (water.isel(y=3) > 0).values  # centre row = the channel
segment_lengths_m = [30.0] * water.sizes["x"]

result = analyze(
    cube,
    aoi_id="walkthrough_reach",
    config=config,
    inputs=AnalysisInputs(
        drainage=context,
        channel_wet_profiles=channel_wet_profiles,
        channel_segment_lengths_m=segment_lengths_m,
        hydroyear_extent=extent,
        max_water_apsec=max_apsec,
        median_apsec=median_apsec,
    ),
    pixel_size_m=30.0,
)
metrics = result.metrics_table
metrics["metric_family"].value_counts()

## Spatial stratification first: the 4 zones

Before reading the family sections below, it helps to know *where* in the reach a metric applies. HydroFragments' spatial stratification (`hydrofragments.spatial.zones.build_zones`) splits the reach into up to 4 mutually exclusive zones based on how often each pixel holds water ("occurrence"):

- **Zone 1 -- in-channel:** pixels on/adjacent to a real drainage line (only emitted when a drainage layer is supplied; our reach has one, via `fx.walkthrough_channel_context`, but this notebook keeps the zone raster itself drainage-free below for a simpler standalone example -- see the `drainage_mask` parameter in `build_zones` if you want to enable it against your own reach).
- **Zone 2 -- persistent off-channel:** wet more often than `t_persist` (default 50%).
- **Zone 3 -- seasonally-flooded:** wet between `t_season` and `t_persist` (default 10%-50%).
- **Zone 4 -- marginal:** wet less than `t_season` (default 10%), but at least occasionally.

Zone 0 (unlabelled) means the pixel was never wet, or had too few valid observations to classify (`min_valid_obs`).

In [ ]:
occurrence = water.values.astype(bool).mean(axis=0)
max_wet_mask = water.values.astype(bool).any(axis=0)
valid_count = np.full(occurrence.shape, water.sizes["time"])

from hydrofragments.spatial.zones import build_zones

zones = build_zones(
    occurrence,
    max_wet_mask=max_wet_mask,
    valid_count=valid_count,
    min_valid_obs=1,  # our demo only has a handful of relevant months; a real
                      # run should use the default (20) or your own floor
)

fig, ax = plt.subplots(figsize=(8, 2.5))
im = ax.imshow(zones.mask, cmap="viridis", vmin=0, vmax=4)
ax.set_title(f"Zones present: {zones.emitted_zones} (has_zone_1={zones.has_zone_1})")
fig.colorbar(im, ax=ax, label="zone (0 = unclassified)", ticks=[0, 1, 2, 3, 4])
plt.show()

## 1. Extent & Persistence -- how much water, how reliably

`metric_family` in `{"extent", "persistence"}`. **APSEC** answers *how much of the fixed reach is wet right now*; **occurrence** answers *how often a location holds water, out of the times it was actually observed*.

In [ ]:
extent_persistence = metrics[metrics["metric_family"].isin(["extent", "persistence"])]
print("metrics in this family:", sorted(extent_persistence["metric"].unique()))

apsec = extent_persistence[extent_persistence["metric"] == "apsec"].sort_values("date")
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(apsec["date"], apsec["value"], marker=".")
ax.set_ylabel("APSEC (%)")
ax.set_title("Wetted area fraction over 3 years -- note the repeating seasonal cycle")
fig.autofmt_xdate()
plt.show()

occurrence_rows = extent_persistence[extent_persistence["metric"] == "occurrence"]
print("AOI occurrence (% of observed months a pixel is wet, averaged):",
      occurrence_rows["value"].iloc[0] if len(occurrence_rows) else "n/a")

**Reading it:** APSEC rising and falling in a clean repeating pattern is exactly what we built into the fixture -- in a real reach this same shape would tell you the reach is behaving like a normal seasonal system, not trending toward permanent loss of extent. `occurrence` gives you the single AOI-wide reliability number behind that pattern.

## 2. Morphology & Fragmentation -- how the water is shaped and broken up

`metric_family` in `{"morphology", "fragmentation"}`. **Number of pools (N)** and **LPI** (largest patch index) are the first-glance fragmentation signal: one connected water body, or many separate fragments? **AWRe**/**AWMSI** describe pool shape (round-and-compact vs. long-and-thin).

In [ ]:
morph_frag = metrics[metrics["metric_family"].isin(["morphology", "fragmentation"])]
print("metrics in this family:", sorted(morph_frag["metric"].unique()))

n_pools = morph_frag[morph_frag["metric"] == "number_of_pools"].sort_values("date")
lpi = morph_frag[morph_frag["metric"] == "lpi"].sort_values("date")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
ax1.plot(n_pools["date"], n_pools["value"], marker=".", color="tab:orange")
ax1.set_title("Number of pools (N)")
ax2.plot(lpi["date"], lpi["value"], marker=".", color="tab:green")
ax2.set_title("LPI (% of wetted area in the largest pool)")
for ax in (ax1, ax2):
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

**Reading it:** watch N and LPI move together. When the channel dries and the two pools separate, N should read 2 and LPI should drop below 100% (the largest pool no longer holds *all* the wetted area); when the channel wets and the pools merge, N drops back to 1 and LPI returns to 100%. This is the connected/fragmented alternation the fixture was built to demonstrate.

## 3. Clustering & Connectivity -- how pools relate in space and network

`metric_family == "clustering"` today: **inter-pool gap** (the primary metric here -- the along-channel distance between separate wet segments) is live in `analyze()`'s output. NNI (nearest-neighbour index, exploratory), RC, TCF, and DCI are part of HydroFragments' broader connectivity design (see `hydrofragments.schema.MetricFamily.CONNECTIVITY` and `hydrofragments.metrics.registry`'s `realised_connectivity`/`tcf` entries) but are gated behind `FIXED_NODES`/`GRAPH` dependencies that `analyze()`'s current `AnalysisInputs` has no way to supply -- so no `metric_family == "connectivity"` rows appear in real output yet. This is a genuine current-state gap, not a notebook omission.

In [ ]:
clustering = metrics[metrics["metric_family"] == "clustering"]
print("metrics in this family:", sorted(clustering["metric"].unique()))

gap_mean = clustering[
    (clustering["metric"] == "inter_pool_gap") & (clustering["statistic"] == "mean")
].sort_values("date")
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(gap_mean["date"], gap_mean["value"], marker=".", color="tab:red")
ax.set_ylabel("mean inter-pool gap (km)")
ax.set_title("Along-channel gap between pools -- 0 when the channel connects them")
fig.autofmt_xdate()
plt.show()

**Reading it:** the gap should collapse toward (or reach) zero exactly when the connecting channel row is wet -- i.e. in the wettest months, matching the N/LPI story above. A persistently large gap across an entire hydrological year, in a real reach, is the signature of a refuge becoming isolated from the rest of the network.

## 4. Dynamics -- how it changes through the hydrological year

`metric_family == "dynamics"`. **Extent contraction** (dry-down rate) answers *how fast is the reach losing wetted area, month over month, within a hydrological year*. This section needs hydrological-year anchors (derived from `hydroyear_extent`) and both a max-water and a median APSEC composite -- see the caveat in `fx.walkthrough_apsec_composites`'s docstring: our composites are a synthetic stand-in to exercise the code path, not a real dual-composite measurement.

In [ ]:
dynamics = metrics[metrics["metric_family"] == "dynamics"]
print("metrics in this family:", sorted(dynamics["metric"].unique()))
dynamics[["hy", "monthly_composite", "value", "unit", "hy_confidence"]].sort_values("hy")

**Reading it:** a negative `extent_contraction` value means the reach is *losing* wetted area over that hydrological year's dry-down period (percent of reach area per month); `composite_sensitive` (in `warning_flags`, not shown above) flags hydrological years where the max-water vs. median composite choice materially changes the answer -- always check it before reporting a dry-down rate as a single number.

## Next steps

- Re-run this notebook against your own reach by swapping `fx.walkthrough_water_timeseries()`/`fx.walkthrough_channel_context()` for `open_water_cube(your_path)` and your own drainage layer.
- For the real DEA/WaterMask-TSFill data path, see `02_dea_via_tsfill.ipynb`.
- For the full adapter/config contract, see [`docs/input_format.md`](../docs/input_format.md).